In [1]:
# Load the text sample
raw_text = ""
with open("the-verdict.txt", encoding="utf-8") as file:
    raw_text = file.read()


def peek(lst, first_n=30):
    if isinstance(lst, list):
        print(f"len {len(lst)}: {lst[:first_n]}")
    else:
        peek(list(lst))


peek(raw_text)

len 20479: ['I', ' ', 'H', 'A', 'D', ' ', 'a', 'l', 'w', 'a', 'y', 's', ' ', 't', 'h', 'o', 'u', 'g', 'h', 't', ' ', 'J', 'a', 'c', 'k', ' ', 'G', 'i', 's', 'b']


In [2]:
# A tokenizer with `re.split`
import re


def tokenize(text: str):
    tokenizer_re = re.compile(r'([,.:;?_!"()\']|--|\s)')
    result = re.split(tokenizer_re, text)
    return [t.strip() for t in result if t.strip()]


print(tokenize("Hello, world. Is this-- a test?"))

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [3]:
# Preprocess the raw text
preprocessed = tokenize(raw_text)
peek(preprocessed)

len 4690: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [4]:
# Map tokens to IDs
all_words = sorted(set(preprocessed))
peek(all_words, 10)

vocab = {word: id for id, word in enumerate(all_words)}
peek(vocab.items())

len 1130: ['!', '"', "'", '(', ')', ',', '--', '.', ':', ';']
len 1130: [('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4), (',', 5), ('--', 6), ('.', 7), (':', 8), (';', 9), ('?', 10), ('A', 11), ('Ah', 12), ('Among', 13), ('And', 14), ('Are', 15), ('Arrt', 16), ('As', 17), ('At', 18), ('Be', 19), ('Begin', 20), ('Burlington', 21), ('But', 22), ('By', 23), ('Carlo', 24), ('Chicago', 25), ('Claude', 26), ('Come', 27), ('Croft', 28), ('Destroyed', 29)]


In [5]:
# A simple tokenizer
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    # Input raw text, split into tokens, return token ids.
    def encode(self, text) -> list[int]:
        def tokenize(text: str):
            tokenizer_re = re.compile(r'([,.:;?_!"()\']|--|\s)')
            result = re.split(tokenizer_re, text)
            return [t.strip() for t in result if t.strip()]

        processed = tokenize(text)

        return [self.str_to_int[t] for t in processed]

    # Convert a list of ids back to text
    def decode(self, ids: list[int]) -> str:
        text = " ".join((self.int_to_str[i] for i in ids))
        # Removes spaces before the specified punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r"\1", text)
        return text

In [6]:
tokenizer = SimpleTokenizerV1(vocab)

test_text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(test_text)
peek(ids)

test_text_out = tokenizer.decode(ids)
print(test_text_out)

len 21: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [7]:
all_words_ext = sorted(set(preprocessed))
all_words_ext.extend(["<|endoftext|>", "<|unk|>"])

vocab_ext = {word: id for id, word in enumerate(all_words_ext)}
print(list(vocab_ext.items())[-3:])

[('yourself', 1129), ('<|endoftext|>', 1130), ('<|unk|>', 1131)]


In [8]:
# Tokenizer V2 that can process `unknow` and `enf of text`
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    # Input raw text, split into tokens, return token ids.
    def encode(self, text) -> list[int]:
        def tokenize(text: str):
            tokenizer_re = re.compile(r'([,.:;?_!"()\']|--|\s)')
            result = re.split(tokenizer_re, text)
            return [t.strip() for t in result if t.strip()]

        processed = tokenize(text)
        processed = [t if t in self.str_to_int else "<|unk|>" for t in processed]

        return [self.str_to_int[t] for t in processed]

    # Convert a list of ids back to text
    def decode(self, ids: list[int]) -> str:
        text = " ".join((self.int_to_str[i] for i in ids))
        # Removes spaces before the specified punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r"\1", text)
        return text

In [9]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

tokenizer = SimpleTokenizerV2(vocab_ext)

print(text)
print(tokenizer.decode(tokenizer.encode(text)))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [10]:
# Use of BPE
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace."
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
peek(ids)

print(tokenizer.decode(ids))

tiktoken version: 0.12.0
len 20: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [11]:
# Prepare dataset and loader

import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + 1 + max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

In [12]:
# Test the loader
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [13]:
# Token Embedding
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(813)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)
print(embedding_layer.weight.size())  # 6*3

# Fetch one embedding, and a batch embedding
print(embedding_layer(torch.tensor([2])))
print(embedding_layer(input_ids))

Parameter containing:
tensor([[-2.4232, -0.4792,  0.4293],
        [ 1.2971, -1.1339,  0.1127],
        [-1.5061, -0.0563,  1.0167],
        [ 1.8833, -2.5037, -0.6321],
        [-0.2036, -0.2649,  1.4164],
        [-1.0882,  0.1874, -1.1885]], requires_grad=True)
torch.Size([6, 3])
tensor([[-1.5061, -0.0563,  1.0167]], grad_fn=<EmbeddingBackward0>)
tensor([[-1.5061, -0.0563,  1.0167],
        [ 1.8833, -2.5037, -0.6321],
        [-1.0882,  0.1874, -1.1885],
        [ 1.2971, -1.1339,  0.1127]], grad_fn=<EmbeddingBackward0>)


In [14]:
# A more realistic embedding

vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

In [15]:
# Read first batch of data
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [16]:
# Convert token to embedding vector
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)  # each token in converted to an embedding vector

torch.Size([8, 4, 256])


In [17]:
# Create position embedding
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))  # [0, context_length)
print(pos_embeddings.shape)

torch.Size([4, 256])


In [18]:
# The final result
input_embedding = token_embeddings + pos_embeddings
print(input_embedding.shape)

torch.Size([8, 4, 256])
